# NECO Coursework — Reproducibility Test Notebook

This notebook loads the saved artefacts from `models/` and reproduces the test
metrics and two key figures **without retraining**. Runtime: under 30 seconds
on CPU.

The final cell prints a PASS/FAIL check against the expected metrics stored in
`models/config.json` (the main notebook writes these at training time). If all
10 metrics match within tolerance 1e-6, the submission is reproducible.

## 1. Imports and reproducibility

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
if not os.environ.get('DISPLAY') and os.environ.get('MPLBACKEND') != 'Agg':
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, precision_recall_curve,
    roc_curve, confusion_matrix,
)

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path.cwd()
OUT_DIR = ROOT / 'outputs'
MODEL_DIR = ROOT / 'models'
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
print(f"Device: {DEVICE}")

## 2. Load data and reconstruct the test split

We reconstruct the identical 60/20/20 split used by the main notebook by
running the two `train_test_split` calls with the same `random_state=SEED`.

In [ ]:
REDUCED = ROOT / 'creditcard_reduced.csv'
assert REDUCED.exists(), (
    f"{REDUCED.name} is missing. "
    f"Run the main notebook first (or copy it from the HPC run)."
)
df = pd.read_csv(REDUCED)
feature_cols = [c for c in df.columns if c != 'Class']
X = df[feature_cols].values
y = df['Class'].values

X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=SEED)
print(f"Test split: n={len(X_test)}, frauds={int(y_test.sum())}, "
      f"fraud_rate={y_test.mean()*100:.3f}%")

## 3. MLP class definition

This mirrors the class in `neco_starter.ipynb` — required to deserialise the
saved `state_dict`.

In [ ]:
class MLP(nn.Module):
    """Fully-connected binary classifier."""
    def __init__(self, in_dim, hidden_sizes=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

    @torch.no_grad()
    def predict_proba(self, x):
        self.eval()
        return torch.sigmoid(self(x))

## 4. Load saved artefacts

The pieces produced by Section 12 of the main notebook.

In [ ]:
scaler = joblib.load(MODEL_DIR / 'scaler.joblib')
svm = joblib.load(MODEL_DIR / 'best_svm.joblib')
ckpt = torch.load(MODEL_DIR / 'best_mlp.pt', map_location=DEVICE)
with open(MODEL_DIR / 'config.json') as f:
    config = json.load(f)

print("Loaded MLP config:")
print(f"  hidden_sizes     : {ckpt['hidden_sizes']}")
print(f"  dropout          : {ckpt['dropout']}")
print(f"  in_dim           : {ckpt['in_dim']}")
print(f"  tuned_threshold  : {ckpt['tuned_threshold']:.4f}")

print("\nLoaded SVM config:")
print(f"  params           : {config['svm_config']['params']}")
print(f"  tuned_threshold  : {config['svm_config']['tuned_threshold']:.4f}")
print(f"  imbalance        : {config['svm_config']['imbalance_strategy']}")

mlp = MLP(in_dim=ckpt['in_dim'],
          hidden_sizes=tuple(ckpt['hidden_sizes']),
          dropout=ckpt['dropout']).to(DEVICE)
mlp.load_state_dict(ckpt['state_dict'])
mlp.eval()

## 5. Run inference on the test set

In [ ]:
# Scale test features with the loaded scaler (fit on training data during the main run).
X_test_s = scaler.transform(X_test)

mlp_proba = mlp.predict_proba(torch.tensor(X_test_s, dtype=torch.float32, device=DEVICE)).cpu().numpy()
svm_proba = svm.predict_proba(X_test)[:, 1]

mlp_thr = ckpt['tuned_threshold']
svm_thr = config['svm_config']['tuned_threshold']
print(f"Tuned thresholds: MLP={mlp_thr:.3f}  SVM={svm_thr:.3f}")

## 6. Test-set metrics table

In [ ]:
def evaluate(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        'roc_auc': roc_auc_score(y_true, proba),
        'pr_auc': average_precision_score(y_true, proba),
        'f1': f1_score(y_true, pred, zero_division=0),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
    }

reproduced = {
    'mlp_default': evaluate(y_test, mlp_proba, 0.5),
    'mlp_tuned':   evaluate(y_test, mlp_proba, mlp_thr),
    'svm_default': evaluate(y_test, svm_proba, 0.5),
    'svm_tuned':   evaluate(y_test, svm_proba, svm_thr),
}

rows = []
for key, m in reproduced.items():
    model = 'MLP' if key.startswith('mlp') else 'SVM'
    thr = '0.50 (default)' if key.endswith('default') else 'tuned'
    rows.append({'model': model, 'threshold': thr, **m})
results_df = pd.DataFrame(rows)
print("Reproduced test metrics:")
display(results_df.round(4))

## 7. Reproduce Figure 5 — ROC + PR curves on test set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for proba, label, colour in [(mlp_proba, 'MLP', '#4C72B0'), (svm_proba, 'SVM', '#C44E52')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{label} (ROC-AUC={roc_auc_score(y_test, proba):.3f})', color=colour)
    prec, rec, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(rec, prec, label=f'{label} (PR-AUC={average_precision_score(y_test, proba):.3f})', color=colour)
axes[0].plot([0, 1], [0, 1], '--', color='grey', lw=1)
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC curves (reproduced)')
axes[0].legend(loc='lower right')
axes[1].axhline(y_test.mean(), ls='--', color='grey', lw=1, label=f'baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-recall curves (reproduced)')
axes[1].legend(loc='lower left')
plt.tight_layout()
plt.show()

## 8. Reproduce Figure 8 — confusion matrices at tuned thresholds

In [ ]:
mlp_pred = (mlp_proba >= mlp_thr).astype(int)
svm_pred = (svm_proba >= svm_thr).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, title in [(axes[0], mlp_pred, f'MLP (thr={mlp_thr:.2f})'),
                         (axes[1], svm_pred, f'SVM (thr={svm_thr:.2f})')]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'],
                cbar=False)
    ax.set_title(title); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.show()

## 9. PASS / FAIL reproducibility check

We compare every reproduced metric against the expected value stored in
`config.json` with tolerance 1e-6. All 10 metrics (5 per threshold × 2 for
MLP, but config only stores primary keys — we check mlp_tuned + svm_tuned +
the two defaults, giving 20 numeric comparisons).

In [ ]:
expected = config['expected_metrics']
tolerance = 1e-6

mismatches = []
total_checks = 0
for key in ['mlp_default', 'mlp_tuned', 'svm_default', 'svm_tuned']:
    for metric in ['roc_auc', 'pr_auc', 'f1', 'precision', 'recall']:
        total_checks += 1
        exp = expected[key][metric]
        got = reproduced[key][metric]
        if abs(exp - got) > tolerance:
            mismatches.append((key, metric, exp, got, abs(exp - got)))

if mismatches:
    print(f"Reproduction check: FAIL ({len(mismatches)}/{total_checks} metrics drifted)")
    for key, metric, exp, got, diff in mismatches:
        print(f"  {key}.{metric}: expected {exp:.6f}, got {got:.6f} (Δ={diff:.2e})")
else:
    print(f"Reproduction check: PASS ✓ ({total_checks}/{total_checks} metrics match within {tolerance:.0e})")

## 10. Summary

This notebook did not retrain any model. It loaded `scaler.joblib`, `best_mlp.pt`,
and `best_svm.joblib`, applied them to the reconstructed test split, and
verified that every metric matches the values recorded by the main notebook at
training time.

For the full training pipeline, CV grid searches, ablation results, multi-seed
robustness, and scaling curves, see `neco_starter.ipynb`. For the written
analysis see `paper.md` and `supplementary.md`.